# Inference time per image

For each setting, we measure the average wall-clock time per image. NPE (settings 1-5) is timed with `batch_size=1` on GPU. AnaCal (settings 1-2) is timed under the same parallel pattern as `run_anacal.py` (`ProcessPoolExecutor` with 28 workers) on CPU. Both methods use 56 images per setting.

In [1]:
import gc
import time
from concurrent.futures import ProcessPoolExecutor

import lightning as L
import numpy as np
import pandas as pd
import torch
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate

from images_to_maps.descwl.run_anacal import process_file

# AnaCal is CPU-only; this device is used only for the NPE encoder.
# Note: PyTorch may still create a small CUDA context on cuda:0 just by initializing
# the CUDA driver. To fully isolate to a single GPU, launch with
# CUDA_VISIBLE_DEVICES=N (then npe_device should be cuda:0).
npe_device = torch.device("cuda:4" if torch.cuda.is_available() else "cpu")
if npe_device.type == "cuda":
    torch.cuda.set_device(npe_device)
print(f"NPE device: {npe_device}  (AnaCal runs on CPU)")

/home/twhit/blissWL/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


NPE device: cuda:4  (AnaCal runs on CPU)


In [2]:
n_warmup = 1
n_timed = 56
n_workers_parallel = 28  # matches config_run_anacal.yaml
replicate = 1
test_index_start = 4000  # test split is indices 4000-4999 per config_train_npe.yaml

data_root = "/nfs/turbo/lsa-regier/scratch/descwl"
ckpt_root = f"{data_root}/checkpoints"

# Setting 2 has variable PSF; AnaCal reconstructs it from the seed config
anacal_setting2_config = {
    "seed": 2,
    "num_images": 14000,
    "variation_factor": 1.0,
    "coadd_dim": 2550,
    "rotate": False,
    "variable_psf": True,
    "pixel_scale": 0.2,
    "npix": 64,
}

anacal_base_cfg = {
    "npix": 64,
    "sigma_arcsec": 0.52,
    "mag_zero": 30.0,
    "pixel_scale": 0.2,
    "combine_method": "inverse_variance",
    "mask_combine_method": "union",
}

In [3]:
def load_samples(setting, n=n_warmup + n_timed):
    """Load `n` test samples (with file paths) for the given setting/replicate=1."""
    samples = []
    for i in range(test_index_start, test_index_start + n):
        path = f"{data_root}/setting{setting}_{replicate}/dataset_{i}_size_1.pt"
        sample = torch.load(path, weights_only=False)[0]
        samples.append((sample, path))
    return samples

## NPE

In [4]:
DESCWL_CONFIG_DIR = "/home/twhit/blissWL/images_to_maps/descwl"


def time_npe(setting, samples):
    """Time NPE encoder forward pass on each pre-loaded sample with batch_size=1.

    Returns a list of per-image wall times (seconds), excluding the warmup pass.
    """
    ckpt = f"{ckpt_root}/trained_encoder_{setting}_{replicate}.ckpt"
    cached_data = f"{data_root}/setting{setting}_{replicate}"

    GlobalHydra.instance().clear()
    with initialize_config_dir(config_dir=DESCWL_CONFIG_DIR, version_base=None):
        hydra_cfg = compose(
            "config_train_npe",
            overrides=[
                f"train.pretrained_weights={ckpt}",
                f"paths.cached_data={cached_data}",
            ],
        )

    L.seed_everything(hydra_cfg.train.seed)

    data_source = instantiate(hydra_cfg.train.data_source)
    data_source.setup("test")
    transform = data_source.test_dataset.transform

    encoder = instantiate(hydra_cfg.encoder).to(npe_device)
    state_dict = torch.load(ckpt, map_location=npe_device)["state_dict"]
    encoder.load_state_dict(state_dict)
    encoder = encoder.eval()

    times = []
    with torch.no_grad():
        for j, (sample, _) in enumerate(samples):
            transformed = transform(sample)
            images = transformed["images"].unsqueeze(0).to(npe_device)

            if npe_device.type == "cuda":
                torch.cuda.synchronize(npe_device)
            t0 = time.perf_counter()
            input_lst = [
                inorm.get_input_tensor({"images": images})
                for inorm in encoder.image_normalizers
            ]
            inputs = torch.cat(input_lst, dim=2).squeeze(2)
            _ = encoder.net(inputs)
            if npe_device.type == "cuda":
                torch.cuda.synchronize(npe_device)
            t1 = time.perf_counter()

            if j >= n_warmup:
                times.append(t1 - t0)

    del encoder, data_source
    gc.collect()
    if npe_device.type == "cuda":
        torch.cuda.empty_cache()

    return times

In [5]:
npe_times = {}
for s in [1, 2, 3, 4, 5]:
    samples = load_samples(s)
    ts = time_npe(s, samples)
    npe_times[s] = ts
    print(f"NPE setting {s}: {np.mean(ts) * 1000:.2f} ± {np.std(ts) * 1000:.2f} ms (n={len(ts)})")

Seed set to 123123
/home/twhit/blissWL/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


NPE setting 1: 176.79 ± 1.54 ms (n=56)


Seed set to 123123


NPE setting 2: 178.93 ± 1.34 ms (n=56)


Seed set to 123123


NPE setting 3: 179.92 ± 1.69 ms (n=56)


Seed set to 123123


NPE setting 4: 180.75 ± 1.56 ms (n=56)


Seed set to 123123


NPE setting 5: 180.97 ± 1.76 ms (n=56)


## AnaCal (matching `run_anacal.py`)

`run_anacal.py` runs AnaCal with `ProcessPoolExecutor(max_workers=28)` per `config_run_anacal.yaml`. We replicate that pattern here: 56 files split across 28 workers (2 files per worker for steady-state throughput), reporting `total_wall_time / n_files` as per-image time.

In [6]:
def time_anacal_parallel(setting, n_files=n_timed, n_workers=n_workers_parallel):
    """Replicate run_anacal.py's parallel pattern: ProcessPoolExecutor over process_file.

    Returns (total_seconds, per_image_seconds).
    """
    cfg = dict(anacal_base_cfg)
    cfg["setting_config"] = anacal_setting2_config if setting == 2 else None

    file_paths = [
        f"{data_root}/setting{setting}_{replicate}/dataset_{i}_size_1.pt"
        for i in range(test_index_start, test_index_start + n_files)
    ]
    args_list = [(p, cfg) for p in file_paths]

    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        for _ in ex.map(process_file, args_list):
            pass
    total = time.perf_counter() - t0
    return total, total / n_files

In [7]:
anacal_per_image = {}
for s in [1, 2]:
    total, per_image = time_anacal_parallel(s)
    anacal_per_image[s] = per_image
    print(
        f"AnaCal setting {s}: {total:.2f} s for {n_timed} files "
        f"with {n_workers_parallel} workers → {per_image:.3f} s/image"
    )

AnaCal setting 1: 12.24 s for 56 files with 28 workers → 0.218 s/image
AnaCal setting 2: 150.74 s for 56 files with 28 workers → 2.692 s/image


## Summary

In [8]:
rows = []
for s in [1, 2, 3, 4, 5]:
    rows.append({
        "Setting": s,
        "AnaCal (s/image)": anacal_per_image.get(s, np.nan),
        "NPE (s/image)": np.mean(npe_times[s]) if s in npe_times else np.nan,
    })
df = pd.DataFrame(rows).set_index("Setting")
df

,AnaCal (s/image),NPE (s/image)
Setting,,
1,0.218499,0.176787
2,2.691721,0.178935
3,NaN,0.179922
4,NaN,0.180755
5,NaN,0.180970
